# Spatial transcriptomics of human skin wound healing — student notebook

**ENS de Lyon, M2 — NGS practicals.**
Data: Visium spatial transcriptomics (GEO [GSE241124](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE241124)) and scRNA-seq (GEO [GSE241132](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE241132)) from *Spatiotemporal single-cell roadmap of human skin wound healing* (Liu et al., Cell Stem Cell 2025).

### What you will do
1. Quality control and filtering of Visium spots and genes, normalisation, dimensionality reduction and clustering.
2. Think about **batch effects**, and when *not* to correct them.
3. Analyse the matching scRNA-seq data of the same donor and **annotate cell types** from marker genes.
4. **Annotate the spatial clusters** and **deconvolve** the spots with Cell2location.
5. Read the results: *where* are the cell types, *when*, and *next to whom*?

### How this notebook works
Every code cell starts with a banner that tells you what is expected from you:

| Banner | What it means |
|---|---|
| `RUN` | Just run the cell. Often it calls a function from `ens_helpers.py` that hides repetitive plumbing code; the banner says what the function does (use `help(eh.function_name)` for details). |
| `READ` | Code that is a bit tricky or new. It is shown in full: **run it, but also read it** and be ready to explain what it does. |
| `YOUR TURN` | **You write the code.** Blanks are marked `____`; the banner lists what to do and what you should obtain. |

**Question** cells are for you: write a short answer, we will discuss them together. Boxes titled *click to reveal* contain the discussion — try before you open them!

### Rules of the game
- Run the cells **in order**. If a cell fails, read the error message first.
- Your own results are saved in `my_results/`. The instructor's checkpoints in `data/processed_data/` are a **safety net**: use them only if you are stuck (they are commented out in the notebook). If you load them, remember that **cluster numbers may differ from yours**: redo the annotation dictionaries with *your* clusters.
- Analysis choices matter more than code. Whenever you pick a threshold or a parameter, be ready to justify it.

In [ ]:
# ==========================================================================
# RUN  |  Libraries
# Nothing to modify. If `import ens_helpers` fails, the file ens_helpers.py is not in the same folder as this notebook.
# ==========================================================================
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import scanpy as sc
import squidpy as sq
import anndata as ad
import spatialdata as sd
import spatialdata_io
import spatialdata_plot
import seaborn as sns

from scipy.stats import spearmanr
from matplotlib.colors import Normalize

import ens_helpers as eh   # helper functions of this practical (ens_helpers.py, next to this notebook)

In [ ]:
# ==========================================================================
# RUN  |  Paths and constants  (the only cell where you must edit something)
# Set BASE to the course folder. Everything else is derived from it.
# If a path error appears later on, this is the first place to check.
# ==========================================================================
BASE = "/path/to/course/folder"      # <-- EDIT: the folder that contains the `data/` directory

DATA        = f"{BASE}/data/GSE241124_RAW"                 # Visium: one folder per sample
SC_DATA     = f"{BASE}/data/GSE241132_RAW"                 # scRNA-seq: one folder per sample
PROC        = f"{BASE}/data/processed_data"                # instructor checkpoints (read only, safety net)
MARKERS_CSV = f"{BASE}/data/paper_marker_genes.csv"        # marker genes reported in the paper
PAPER_META  = f"{SC_DATA}/GSE241132_cell_metadata.txt"     # per-cell metadata published by the authors
Q05_FILE    = f"{PROC}/q05_cell_abundance.csv"             # Cell2location output (computed by the instructors)

MY = f"{BASE}/my_results"                                  # YOUR checkpoints are written here
os.makedirs(MY, exist_ok=True)

SAMPLES = ["Skin", "Wound1", "Wound7", "Wound30"]          # the 4 Visium sections (same donor, 4 timepoints)
METRICS = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
TITLES = {
    "total_counts": "Total UMIs per spot",
    "n_genes_by_counts": "Detected genes per spot",
    "pct_counts_mt": "% mitochondrial counts per spot",
}

sns.set_style("whitegrid")

## Spatial Transcriptomics data analysis

### Loading the Visium data

We analyse the sections of **one donor** at four timepoints: healthy skin (`Skin`) and wound-edge biopsies at day 1, 7 and 30 (`Wound1`, `Wound7`, `Wound30`). Each section is read into a [SpatialData](https://spatialdata.scverse.org) object, which keeps together the expression table, the spot coordinates (shapes) and the tissue images.

In [ ]:
# ==========================================================================
# READ  |  Load one SpatialData object per sample
# Nothing to modify. `spatialdata_io.visium` reads a Space Ranger output folder.
# ==========================================================================
sdatas = {
    s: spatialdata_io.visium(f"{DATA}/{s}", dataset_id=s)
    for s in SAMPLES
}

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 1  -  Inspect a SpatialData object
# ==========================================================================
# What to do:
#   1. Display one object, e.g. sdatas["Skin"]. Which elements does it contain (images, shapes, tables)?
#   2. Look at its table: how many spots and genes? What is in .obs? Where are the spot coordinates stored (.obsm)?
# Expected: you can answer the question below.

# write your code here

**Question 1.** What does each element of the SpatialData object contain, and what links the expression table to the spots drawn on the image?

*Your answer (double-click to edit):* 

### Quality control

We compute, for every spot, three standard QC metrics: the **total number of UMIs**, the **number of detected genes** and the **percentage of mitochondrial counts**. The function `eh.qc_metrics` flags the mitochondrial genes (symbol starting with `MT-`) and calls `sc.pp.calculate_qc_metrics`.

In [ ]:
# ==========================================================================
# RUN  |  QC metrics for every sample
# Expected: one line per sample with the number of spots and genes.
# ==========================================================================
adatas = {}
for s in SAMPLES:
    table = sdatas[s]["table"].copy()
    eh.qc_metrics(table)            # adds total_counts, n_genes_by_counts, pct_counts_mt to table.obs
    table.obs["sample"] = s
    adatas[s] = table
    print(f"{s}: {table.n_obs} spots, {table.n_vars} genes")

In [ ]:
# ==========================================================================
# RUN  |  QC violin plots, one figure per sample
# Expected: 4 figures with 3 panels each (UMIs, genes, % mitochondrial).
# Look carefully: you will need these plots to choose your filtering thresholds below.
# ==========================================================================
eh.plot_qc_violins(adatas, METRICS, TITLES, unit="spots")

In [ ]:
# ==========================================================================
# READ  |  Combine the four samples into one object and compare them
# Run it and look at the columns of table_all.obs.
# Note the `region` column: it is created by spatialdata_io and contains the sample name.
# ==========================================================================
# sd.concatenate merges the SpatialData objects. concatenate_tables=True also concatenates
# the underlying tables, so all spots of all samples end up in one shared table.
sdata_all = sd.concatenate(list(sdatas.values()), concatenate_tables=True)
table_all = sdata_all["table"].copy()      # the merged expression table
eh.qc_metrics(table_all)                    # QC metrics on the merged table
table_all.obs.head()

In [ ]:
# ==========================================================================
# RUN  |  QC of the four samples side by side
# Expected: 3 panels (one per metric), 4 violins per panel.
# ==========================================================================
eh.plot_qc_combined(table_all.obs, "region", SAMPLES, METRICS, TITLES, "QC combined across samples")

**Question 2.** Compare the four samples. Do they have the same sequencing depth? Which sample looks different, and what could explain it?

*Your answer (double-click to edit):* 

Visualize the metrics on the tissue, by using the squidpy command sq.pl.spatial_scatter()
If we worked with anndata structures, something like this would work:

```python
for s in SAMPLES:
    adata = adatas[s]
    sq.pl.spatial_scatter(adata, color="log1p_n_genes_by_counts")
    sq.pl.spatial_scatter(adata)
```

But spatialdata_io.visium() does not save images in the classical structure adata.uns["spatial"][library_id]["images"] that sq.pl.spatial_scatter() expects. In a spatialdata structure, images are separated elements, e.g., sdata.images["Skin_hires_image"], and not inside a .uns slot. 

Then, in order to use sq.pl.spatial_scatter(), we need to reconstruct the demanded structure.
It will be done through a custom function in the cell below.

Note: you are not obliged to do that, this is not a fundamental part of the analysis, it is just a further visual inspection of our data.

*In this practical the function that rebuilds this structure (`eh.attach_spatial_uns`) is provided: you do not have to write it.*

In [ ]:
# ==========================================================================
# RUN  |  Visualise the QC metrics on the tissue
# Expected: a grid with one row per sample: tissue image | detected genes | total UMIs.
# Is low quality random in space, or does it follow tissue structures?
# ==========================================================================
for s in SAMPLES:
    eh.attach_spatial_uns(adatas[s], sdatas[s], s, DATA)    # rebuild adata.uns["spatial"] for squidpy

eh.plot_spatial_qc(adatas, SAMPLES)

### Filtering spots

Spots with very few detected genes are mostly empty or poorly captured spots. Where to cut is a judgement call that **depends on the sample** (the sequencing depth is not the same): use the per-sample violin plots above.

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 2  -  Choose the minimum number of detected genes per sample
# ==========================================================================
# What to do:
#   Replace each None with the minimum number of detected genes (n_genes_by_counts) that a spot must have
#   in that sample, by looking at where the low-quality tail starts in the violin plots.
#   Write one sentence below justifying your choice for each sample.
# Expected: a few % of spots removed per sample, not 50%. Check the printout.

MIN_GENES_PER_SAMPLE = {
    "Skin": None,
    "Wound1": None,
    "Wound7": None,
    "Wound30": None,
}
assert all(v is not None for v in MIN_GENES_PER_SAMPLE.values()), "Fill in a threshold for every sample"

for s in SAMPLES:
    n_before = adatas[s].n_obs
    sc.pp.filter_cells(adatas[s], min_genes=MIN_GENES_PER_SAMPLE[s])
    n_after = adatas[s].n_obs
    print(
        f"{s}: min_genes={MIN_GENES_PER_SAMPLE[s]} -> "
        f"{n_before} to {n_after} spots ({n_before - n_after} removed, "
        f"{(n_before - n_after) / n_before * 100:.1f}%)"
    )

*Your justification (double-click to edit):* 

### Filtering genes, collectively across samples

We now put the four filtered samples into one table and remove genes that are detected in almost no spot.

In [ ]:
# ==========================================================================
# READ  |  Concatenate the filtered samples
# Nothing to modify. Read the comments: what would `join="outer"` do instead?
# ==========================================================================
table_all = ad.concat(
    [adatas[s] for s in SAMPLES],
    join="inner",        # only keep the genes present in all samples
    index_unique="-",    # make the spot barcodes unique by appending the sample key
    keys=SAMPLES,        # one key per concatenated object
)

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 3  -  Remove genes detected in very few spots
# ==========================================================================
# What to do:
#   Keep only the genes detected in at least 3 spots (sc.pp.filter_genes, argument min_cells).
# Expected: a printout with the number of genes before/after, the table shape and the spots per sample.

n_genes_before = table_all.n_vars
____                                    # <- your filtering call
print(f"Genes: {n_genes_before} -> {table_all.n_vars} ({n_genes_before - table_all.n_vars} removed)")
print(f"Combined table: {table_all.shape[0]} spots x {table_all.shape[1]} genes")
print(table_all.obs["sample"].value_counts())

### Do we need to filter spots on the percentage of mitochondrial counts?

In dissociated single-cell data, a high mitochondrial fraction is a classic sign of dying cells, and cells above ~20% are usually removed. Does the same logic apply to a **tissue section**?

In [ ]:
# ==========================================================================
# RUN  |  How many spots would a 20% mitochondrial threshold remove?
# Nothing to modify. We only look at the number: no spot is removed.
# ==========================================================================
# demo only - NOT applied to the data
n_would_remove = (table_all.obs["pct_counts_mt"] > 20).sum()
print(
    f"Spots with pct_counts_mt > 20%: {n_would_remove} / {table_all.n_obs} "
    f"({n_would_remove / table_all.n_obs * 100:.2f}%)"
)

**Question 3.** Based on this number and on what a Visium spot is, would you filter on `pct_counts_mt` in this dataset? Why (not)? And how could a very active tissue region affect this metric?

*Your answer (double-click to edit):* 

<details>
<summary><b>Discussion: mitochondrial counts in tissue, and the strategy of the paper</b> (try on your own first, then click)</summary>

In this dataset that threshold would remove almost nothing. In **intact tissue** (as opposed to a dissociated single-cell suspension), a spot's mitochondrial fraction does not primarily reflect membrane damage: it can just as well reflect a metabolically very active tissue region (e.g. proliferative/inflammatory zones during wound healing).
Filtering spots on `pct_counts_mt` therefore risks discarding real biological signal rather than technical noise — and, as we just saw, it removes very few spots anyway, so it would not have meaningfully improved data quality here.

This is exactly the reasoning behind the QC strategy used in the original publication for their ST-seq data. Rather than filtering spots on mitochondrial content, they removed three families of highly and uniformly
expressed, but largely uninformative, genes **from the expression matrix itself**, collectively across all sections:

- **`MALAT1`** — an extremely abundant, ubiquitously expressed nuclear lncRNA.
  Because its expression barely varies between cell types, it contributes
  little discriminative signal, but its very high, variable capture
  efficiency (driven by nuclear retention and technical factors rather than
  biology) can dominate normalization and distort clustering/PCA if left in.
- **Mitochondrial genes** — rather than using `pct_counts_mt` as a
  spot-exclusion criterion (which, as shown above, is not a reliable quality
  signal in intact tissue), the mitochondrial genes themselves are removed
  from the matrix. Their expression reflects metabolic state / technical
  capture bias more than cell identity, and can otherwise add nuisance
  variation to HVG selection and PCA.
- **Hemoglobin genes** (`HBA1`, `HBA2`, `HBB`, `HBD`, `HBE1`, `HBG1`, `HBG2`,
  `HBM`, `HBQ1`, `HBZ`) — markers of red blood cell contamination. Wound
  tissue is prone to local bleeding, so some spots capture a mix of real
  tissue and red blood cells (or their ambient RNA). Left in, these genes can
  dominate the profile of an otherwise normal spot and create spurious
  "erythrocyte-like" clusters.

We apply this filter collectively (not per sample).

</details>

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 4  -  Remove MALAT1, mitochondrial and hemoglobin genes
# ==========================================================================
# What to do:
#   Following the paper, remove from the expression table (collectively, not per sample):
#     - MALAT1
#     - all mitochondrial genes (gene symbol starting with "MT-")
#     - the hemoglobin genes listed below (only those that exist in the table)
#   Hint: table_all.var_names.str.startswith(...), .isin(...), and don't forget .copy().
#   PITFALL: do NOT select hemoglobin genes with a prefix like "HB": you would also remove HBEGF,
#   a real wound-healing growth factor (and HBP1, HBS1L). Use the explicit list.
# Expected: a few dozen genes removed at most.

HEMOGLOBIN_GENES = [
    "HBA1", "HBA2", "HBB", "HBD", "HBE1",
    "HBG1", "HBG2", "HBM", "HBQ1", "HBZ",
]

mt_genes = ____            # list of mitochondrial genes present in the table
hb_genes = ____            # hemoglobin genes present in the table
malat1_gene = ____         # ["MALAT1"] if present
genes_to_remove = set(malat1_gene) | set(mt_genes) | set(hb_genes)

print(f"Mitochondrial genes ({len(mt_genes)}): {mt_genes}")
print(f"Hemoglobin genes ({len(hb_genes)}): {hb_genes}")
print(f"Total genes removed: {len(genes_to_remove)}")

n_genes_before = table_all.n_vars
table_all = ____           # your filtered table (a copy)
print(f"Genes: {n_genes_before} -> {table_all.n_vars}")
print(f"Final table_all: {table_all.shape[0]} spots x {table_all.shape[1]} genes")

Put back all the info in the spatialdata object and save it

In [ ]:
# ==========================================================================
# RUN  |  Put the filtered table back in the SpatialData object and save it
# Nothing to modify. It writes a checkpoint in your my_results/ folder.
# ==========================================================================
sdata_all.tables["table"] = table_all      # replaces the original table by the filtered one
sdata_all.write(f"{MY}/spatial_data_filtered.zarr", overwrite=True)

In [ ]:
# ==========================================================================
# RUN  |  Reload the checkpoint
# Expected: the shape of the table and the 4 sample names, then the SpatialData summary.
# ==========================================================================
sdata_all = sd.read_zarr(f"{MY}/spatial_data_filtered.zarr")
# --- SAFETY NET: if you are stuck above, load the instructor's checkpoint instead (uncomment) ---
# sdata_all = sd.read_zarr(f"{PROC}/spatial_data_filtered.zarr")

adata = sdata_all["table"]
# spatialdata_io calls the sample column "region", which is misleading here (it is always the same
# wound, at different timepoints). We copy it to a column called "sample", used from now on.
adata.obs["sample"] = adata.obs["region"].astype(str)
print(adata.shape, adata.obs["sample"].unique().tolist())
sdata_all

### Normalisation, highly variable genes, PCA

In [ ]:
# ==========================================================================
# READ  |  Keep the raw counts
# Nothing to modify. Never forget this step before normalising!
# ==========================================================================
# keep raw counts: needed later by Cell2location, which requires raw (not normalised) counts
adata.layers["counts"] = adata.X.copy()

We normalize with a standard total-count + log1p approach. Highly variable genes are selected in a batch-aware way (batch_key="sample") so that genes are not selected merely because they differ between samples

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 5  -  Normalisation, highly variable genes, PCA
# ==========================================================================
# What to do (3 steps, in this order):
#   1. Normalise each spot to 10,000 total counts, then log-transform      -> sc.pp.normalize_total, sc.pp.log1p
#   2. Select 2,000 highly variable genes, batch-aware (batch_key="sample") -> sc.pp.highly_variable_genes
#   3. PCA with 30 components (it uses the highly variable genes by default) -> sc.tl.pca
# Expected: adata.var["highly_variable"] has ~2000 True values, adata.obsm["X_pca"] has 30 columns.

____
____
____

print(adata.obsm["X_pca"].shape, int(adata.var["highly_variable"].sum()))

### Uncorrected UMAP, colored by sample

In [ ]:
# ==========================================================================
# READ  |  Neighbors graph and UMAP, without batch correction
# Nothing to modify. Note `key_added`: it lets us keep several neighbor graphs in the same object.
# ==========================================================================
sc.pp.neighbors(adata, n_pcs=30, key_added="neighbors_uncorrected")
sc.tl.umap(adata, neighbors_key="neighbors_uncorrected")
# Each new UMAP overwrites adata.obsm["X_umap"]: we store a copy of this one under another name.
adata.obsm["X_umap_uncorrected"] = adata.obsm["X_umap"].copy()

sc.pl.embedding(adata, basis="X_umap_uncorrected", color="sample", title="uncorrected")

### Batch effects: to correct or not to correct?

The four samples separate on the UMAP above. Four different datasets put together, a clear separation… the textbook reflex is to *remove the batch effect*.

**Question 4.** Before correcting anything: what exactly does the variable `sample` represent in this dataset? Is it a technical batch, a biological condition, or both? What would you lose by correcting for it?

*Your answer (double-click to edit):* 

<details>
<summary><b>Discussion: batch and biology are confounded here</b> (try on your own first, then click)</summary>

In general, looking at the UMAP above, and taking into account that we have 4 different datasets that we put together, we might want to try to delete the batch effect from our data. 
However, this is the WRONG thing to do here: with these 4 samples, each is a single donor at a given timepoint. Timepoint (sample) is perfectly confounded with the technical batch (there is no replicate of the same timepoint processed as a separate technical batch). Correcting for sample as if it were a technical batch effect therefore risks removing exactly the biological signal we want to study (the wound healing dynamics), not technical noise.

The statistically correct batch key would be donor, using a dataset with multiple donors per timepoint (the full study has 4 donors for ST-seq), that would let us remove donor-to-donor technical variability while preserving the timepoint signal. We don't do this here for time reasons, but the message I want to deliver is: always check what your "batch" variable actually represents before correcting for it.

</details>

Now, just to show you what are 3 among the most common tools to remove for the batch effect, we will remove it by using Harmony, BBKNN and Scanorama (they are conceptually very different, but with the same practical purpose).

We will see that the three UMAPs are very different (and this could open a big discussion).

But for the rest of this practical, we will proceed with the uncorrected PCA/UMAP.

In [ ]:
# ==========================================================================
# READ  |  BBKNN: correct the neighbor graph
# Nothing to modify. Run it and look at the UMAP.
# ==========================================================================
# --- BBKNN ---
# Corrects the KNN graph directly (balanced neighbors per batch), not the embedding --
# no "corrected PCA" to reuse elsewhere afterwards, only a corrected graph/UMAP.

import bbknn
adata.obsm["X_pca_for_bbknn"] = adata.obsm["X_pca"].copy()
bbknn.bbknn(
    adata, batch_key="sample", use_rep="X_pca_for_bbknn",
    neighbors_within_batch=3, key_added="bbknn",   # <-- non sovrascrive più il default
)
sc.tl.umap(adata, neighbors_key="bbknn")
adata.obsm["X_umap_bbknn"] = adata.obsm["X_umap"].copy()
sc.pl.embedding(adata, basis="X_umap_bbknn", color="sample", title="bbknn")

In [ ]:
# ==========================================================================
# READ  |  Scanorama: correct the expression matrix
# Nothing to modify. It is the slowest of the three: be patient (and read the comments).
# ==========================================================================
# --- Scanorama ---
# Corrects the expression matrix directly (mutual-nearest-neighbor "panorama stitching").
# Heaviest of the three -- expect it to take noticeably longer than Harmony/BBKNN.
import scanorama

samples_list = adata.obs["sample"].unique().tolist()
adatas_split = [adata[adata.obs["sample"] == s].copy() for s in samples_list]
corrected = scanorama.correct_scanpy(adatas_split, return_dimred=True)
adata_scan = ad.concat(corrected, join="inner")

sc.pp.neighbors(adata_scan, use_rep="X_scanorama")
sc.tl.umap(adata_scan)

sc.pl.umap(adata_scan, color="sample", title="scanorama")

In [ ]:
# ==========================================================================
# READ  |  Harmony: correct the PCA embedding
# Nothing to modify. Note the workaround for the broken scanpy wrapper: harmonypy is called directly.
# ==========================================================================
# --- Harmony ---
# NOTE: sc.external.pp.harmony_integrate is currently BROKEN with harmonypy>=2.0
# (shape mismatch: the scanpy wrapper does harmony_out.Z_corr.T assuming an old
# harmonypy convention that no longer holds). Call harmonypy directly instead:

import harmonypy
ho = harmonypy.run_harmony(adata.obsm["X_pca"], adata.obs, ["sample"])
adata.obsm["X_pca_harmony"] = ho.Z_corr
sc.pp.neighbors(adata, use_rep="X_pca_harmony", key_added="harmony")
sc.tl.umap(adata, neighbors_key="harmony")
adata.obsm["X_umap_harmony"] = adata.obsm["X_umap"].copy()

sc.pl.embedding(adata, basis="X_umap_harmony", color="sample", title="harmony")

**Question 5.** The three UMAPs are very different. What does each method correct (graph, embedding, expression)? Do you trust one more than the others? In which situation would correcting for batch be the right thing to do here?

*Your answer (double-click to edit):* 

### Leiden clustering on uncorrected data (no batch correction applied)

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 6  -  Leiden clustering on the uncorrected data
# ==========================================================================
# What to do:
#   1. Run the Leiden algorithm (sc.tl.leiden) on the UNCORRECTED neighbor graph (the one stored as
#      "neighbors_uncorrected"), resolution 0.5, and store the result in adata.obs["leiden"].
#   2. Print the number of spots per cluster.
#   3. Plot the clusters on the uncorrected UMAP (basis "X_umap_uncorrected").
# Expected: about 7 clusters.

____
print(adata.obs["leiden"].value_counts())

____

In [ ]:
# ==========================================================================
# RUN  |  Save a checkpoint with the clusters
# Nothing to modify. Next time, you can restart from here.
# ==========================================================================
sdata_all.tables["table"] = adata
sdata_all.write(f"{MY}/spatial_data_clustered.zarr", overwrite=True)

### Clusters on the tissue

To plot on the tissue we have to make sure that the table is correctly linked to the spots (shapes) of the SpatialData object. This bookkeeping is done by `eh.relink_table` (it re-declares the table annotation, which can be lost when the table is modified).

In [ ]:
# ==========================================================================
# RUN  |  Plot the Leiden clusters on the tissue sections
# Expected: one panel per sample, spots coloured by cluster.
# Do the clusters follow the tissue structures you see in the H&E image?
# ==========================================================================
adata = eh.relink_table(sdata_all, adata, SAMPLES)

fig, axes = plt.subplots(1, len(SAMPLES), figsize=(5 * len(SAMPLES), 5))
eh.spatial_row(sdata_all, "leiden", axes, SAMPLES, cmap=None)
fig.tight_layout()
plt.show()

**Question 6.** Describe how the clusters are organised in space. Do some clusters appear only in certain samples? What does it tell you?

*Your answer (double-click to edit):* 

## Locating the wound within each section

Looking at the H&E image alone, it isn't obvious *where* the wound actually is
in each section, as the four samples don't show an obvious single "wound spot"
you can point to by eye. There's a reason for this: according to the paper's
methods, the `Wound1`/`Wound7`/`Wound30` samples are biopsies taken at the
**wound edge**, not at the wound center. Each section is expected to capture
a *transition* — from more intact skin on one side to actively healing/
regenerating tissue on the other — rather than a single well-defined "wound"
region.

Instead of trying to guess this from tissue morphology alone, we can use gene
expression to locate it directly. During wound healing, keratinocytes at the
migrating epidermal front switch on a specific set of "activation" keratins
that are essentially absent from normal, fully differentiated epidermis:
**KRT6A, KRT6B, KRT6C, KRT16, KRT17**. Alongside these, **S100A8** and
**S100A9** (alarmins associated with inflammation and epithelial stress) are
also strongly induced in this region. None of these genes are expressed at
appreciable levels in intact, quiescent epidermis.

By plotting these genes spatially on the tissue, spots with high expression
should mark the wound-edge/migrating epidermis, giving us a data-driven way
to locate the wound, rather than relying on visual inspection of the H&E
image. This is also a first concrete link between gene expression and tissue
morphology, before we get to formal cluster annotation.

In [ ]:
# ==========================================================================
# RUN  |  Wound-edge marker genes on the tissue
# Expected: 4 figures (one per gene), 4 panels each.
# ==========================================================================
wound_edge_markers = ["KRT6A", "KRT16", "KRT17", "S100A8"]

for gene in wound_edge_markers:
    fig, axes = plt.subplots(1, len(SAMPLES), figsize=(5 * len(SAMPLES), 5))
    eh.spatial_row(sdata_all, gene, axes, SAMPLES)
    fig.suptitle(gene)
    fig.tight_layout()
    plt.show()

**Question 7.** Where is the wound edge in each section? Do the four genes agree with each other? Which timepoint shows the strongest signal, and is it surprising?

*Your answer (double-click to edit):* 

## Single-cell data analysis

### Why we cluster scRNA-seq and ST-seq independently, and how they connect

Before diving into the single-cell data, it's worth being explicit about the
strategy we're following here, because it's easy to misunderstand.

We are **not** going to look for "the same clusters" in the ST data as in
the scRNA-seq data. This is not a matter of tuning parameters, and it is
structurally impossible for the two clusterings to converge to identical
clusterings, because the underlying data are fundamentally different:

- **scRNA-seq**: each row is a single cell. A "pure" transcriptional profile,
  belonging to (at most) one cell type. With tens of thousands of cells per
  donor, we have enough statistical power to support fine-grained, well-
  separated clusters.
- **ST-seq (Visium)**: each row is a *spot*, typically covering several
  cells at once. The signal is already a mixture by construction: e.g., a spot
  placed at a tissue boundary can contain both keratinocytes and
  fibroblasts. There is no resolution setting that turns a mixed signal into
  a pure one.

Because of this, we deliberately keep the two clusterings **separate and
independent**: the ST leiden clustering you've already computed stays as it
is, and the scRNA-seq clustering (about to come, with more care and using
RESOLUTE to choose the resolution rather than guessing) is its own thing.

**scRNA-seq gets the more careful treatment on purpose.** Since each sc
cluster corresponds to a real, relatively pure cell population, it's worth
investing in getting the resolution right, and this clustering will become our
reference of cell-type marker genes.

**Connecting the two datasets** then happens in two complementary ways,
which we'll use in this order:

1. **Marker-gene matching (fast, qualitative)**: once the scRNA-seq
   clusters are annotated (with the help of the paper's marker tables), we
   compare their marker genes against the marker genes (DEGs) of each ST
   cluster. If an ST cluster's top markers overlap strongly with, say, the
   sc "Basal-I" signature, we label that ST cluster accordingly. This is
   exactly the approach described in the paper's methods: *"Spot clusters
   were annotated based on DEGs and markers of distinct cell types from
   scRNA-seq."* It's a cluster-to-cluster comparison based on shared marker
   genes, not a search for identical clusters.

2. **Deconvolution (Cell2location, richer but heavier)**: instead of
   assigning one discrete label per ST cluster, deconvolution estimates,
   for *each individual spot*, the proportion of every sc-derived cell type
   present in it. This properly accounts for the fact that a spot is a
   mixture rather than forcing it into a single category. It's the more
   rigorous, final approach used in the paper, and the one we'll apply if
   time allows.

So: careful, independent clustering on each dataset first, and then two ways of
building a bridge between them, going from the quick/qualitative to the
rigorous/quantitative.

### Loading the scRNA-seq data of the same donor

We use the scRNA-seq samples of donor **PWH27** (the same donor as the Visium sections), at four timepoints: D0 (healthy skin) and D1, D7, D30 after wounding.

In [ ]:
# ==========================================================================
# READ  |  Load and concatenate the four timepoints
# Nothing to modify. Note how the barcodes are renamed: we will need this convention later,
# when we match our cells with the annotation published by the authors.
# ==========================================================================
donor = "PWH27"
timepoints = ["D0", "D1", "D7", "D30"]

adatas_sc = {}
for tp in timepoints:
    a = sc.read_10x_mtx(f"{SC_DATA}/{donor}{tp}")
    a.obs["donor"] = donor
    a.obs["sample"] = tp
    a.obs_names = [f"{bc}-{tp}" for bc in a.obs_names]      # make barcodes unique across timepoints
    adatas_sc[tp] = a

adata_sc = ad.concat(list(adatas_sc.values()), join="inner")
adata_sc

In [ ]:
# ==========================================================================
# RUN  |  QC metrics and violin plots (same as for the spatial data)
# Expected: 4 figures (one per timepoint) and a combined figure.
# ==========================================================================
eh.qc_metrics(adata_sc)

SC_SAMPLES = ["D0", "D1", "D7", "D30"]
SC_TITLES = {
    "total_counts": "Total UMIs per cell",
    "n_genes_by_counts": "Detected genes per cell",
    "pct_counts_mt": "% mitochondrial counts per cell",
}

eh.plot_qc_violins({s: adata_sc[adata_sc.obs["sample"] == s] for s in SC_SAMPLES}, METRICS, SC_TITLES, unit="cells")
eh.plot_qc_combined(adata_sc.obs, "sample", SC_SAMPLES, METRICS, SC_TITLES, "QC combined across samples (donor PWH27)")

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 7  -  Filter low-quality cells
# ==========================================================================
# What to do: apply the same cell-level filters as the paper
#   - at least 500 detected genes per cell          (sc.pp.filter_cells, min_genes)
#   - at least 1000 UMIs per cell                   (sc.pp.filter_cells, min_counts)
#   - less than 20% mitochondrial counts            (boolean mask on adata_sc.obs["pct_counts_mt"], then .copy())
# Note: here (dissociated cells), the mitochondrial filter IS appropriate. Why is it different from the tissue?
# Expected: the number of cells decreases at each step. Print it.

print("before:", adata_sc.n_obs)
____
____
____
print("after:", adata_sc.n_obs)

### Doublets

Two cells captured in the same droplet give a mixed profile that can look like a new cell type. **Scrublet** simulates doublets from the data and scores each cell by its similarity to them.

In [ ]:
# ==========================================================================
# READ  |  Detect and remove doublets with Scrublet
# Nothing to modify. Why do we run it per sample (batch_key)?
# ==========================================================================
sc.pp.scrublet(adata_sc, batch_key="sample")       # run per sample: doublets only form within one droplet batch
print(adata_sc.obs["predicted_doublet"].value_counts())

n_before = adata_sc.n_obs
adata_sc = adata_sc[~adata_sc.obs["predicted_doublet"]].copy()
print(f"Doublet removal: {n_before} -> {adata_sc.n_obs} cells")

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 8  -  Gene filtering
# ==========================================================================
# What to do: same strategy as for the spatial data
#   1. Remove MALAT1, the mitochondrial genes and the hemoglobin genes (reuse HEMOGLOBIN_GENES).
#   2. Then keep only the genes detected in at least 10 cells (sc.pp.filter_genes, min_cells).
# Expected: a few dozen genes removed in step 1, a few thousand in step 2.

n_genes_before = adata_sc.n_vars
____
____
print(f"Genes: {n_genes_before} -> {adata_sc.n_vars}")

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 9  -  Normalisation, PCA, UMAP (uncorrected)
# ==========================================================================
# What to do: repeat the pipeline you wrote for the spatial data, on adata_sc:
#   1. normalise to 10,000 counts + log1p
#   2. 2,000 highly variable genes, batch-aware (batch_key="sample")
#   3. PCA (30 components)
#   4. neighbors graph (30 PCs), then UMAP
#   5. plot the UMAP coloured by "sample"
# Expected: a UMAP in which the timepoints are partially separated.

adata_sc.layers["counts"] = adata_sc.X.copy()      # raw counts, needed later by Cell2location (already written for you)

____

In [ ]:
# ==========================================================================
# RUN  |  Save a checkpoint, and a batch correction demo
# Nothing to modify. Same reasoning as before: with a single donor, timepoint and batch are confounded.
# ==========================================================================
adata_sc.write_h5ad(f"{MY}/sc_data_PWH27_processed.h5ad")

# Batch correction demo with Harmony (same logic and same caveats as for the spatial data)
ho_sc = harmonypy.run_harmony(adata_sc.obsm["X_pca"], adata_sc.obs, ["sample"])
adata_sc.obsm["X_pca_harmony"] = ho_sc.Z_corr
sc.pp.neighbors(adata_sc, use_rep="X_pca_harmony", key_added="harmony")
sc.tl.umap(adata_sc, neighbors_key="harmony")
sc.pl.umap(adata_sc, neighbors_key="harmony", color="sample", title="harmony (single donor)")

# The Harmony UMAP just overwrote adata_sc.obsm["X_umap"]. We continue with the UNCORRECTED data,
# as decided: recompute the UMAP on the default ("neighbors") graph.
sc.tl.umap(adata_sc)
sc.pl.umap(adata_sc, color="sample", title="uncorrected (recomputed)")

### Choosing the clustering resolution

The resolution of Leiden clustering controls how many clusters we get. Instead of guessing, **RESOLUTE** scans a range of resolutions and scores each of them with several criteria (Calinski–Harabasz, BIC, and bootstrap stability).

In [ ]:
# ==========================================================================
# READ  |  RESOLUTE with the Calinski-Harabasz score (and bootstrap stability)
# Nothing to modify. This can take a few minutes. Look at the plots it produces.
# ==========================================================================
import resolute as rs

results_calinski = rs.run_resolute(
    adata=adata_sc,
    score_value="calinski",
    use_rep="X_pca",  # uncorrected data, as decided
    min_res=0.1, max_res=2.0, step=0.1,
    clustering_algorithm="leiden",
)
best_resolution = results_calinski["best_score_res"]

In [ ]:
# ==========================================================================
# READ  |  RESOLUTE with the BIC score
# Nothing to modify. Look at the BIC curve: does it show a clear optimum?
# ==========================================================================
results_bic = rs.run_resolute(
    adata=adata_sc,
    score_value="bic",
    use_rep="X_pca",
    compute_stability=False,       # do not recompute bootstrap stability for UMAP
    min_res=0.1, max_res=2.0, step=0.1,
    clustering_algorithm="leiden",
)

**Question 8.** Compare the three criteria (Calinski-Harabasz, BIC, stability). Do they agree? Which resolution would you choose, and why? Remember that the resolution is a working choice, validated afterwards by the biology.

*Your answer (double-click to edit):* 

<details>
<summary><b>What we chose in our run</b> (try on your own first, then click)</summary>

We used RESOLUTE to test three independent criteria for selecting the leiden clustering resolution: BIC, Calinski-Harabasz, and bootstrap stability.

Two out of three criteria (Calinski-Harabasz and stability) agree on **resolution 0.2** as the optimal choice. BIC instead keeps decreasing across the whole tested range without showing a clear turning point, which suggests it did not find a genuine optimum within these settings.

As a first pass, we proceed with **resolution 0.2**, since it is supported by two independent and conceptually different criteria. We will treat this as a working choice: the real test is whether the resulting clusters map onto coherent, biologically interpretable cell types once we look at marker genes (`rank_genes_groups`), and not the resolution value itself.

</details>

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 10  -  Final Leiden clustering
# Nothing else to write: just fill CHOSEN_RESOLUTION. Expected: ~15-20 clusters at the resolution we used.
# ==========================================================================
# What to do: set CHOSEN_RESOLUTION to the value you decided, then run the cell.
CHOSEN_RESOLUTION = None
assert CHOSEN_RESOLUTION is not None, "Choose a resolution first"

sc.tl.leiden(
    adata_sc,
    neighbors_key="neighbors",
    resolution=CHOSEN_RESOLUTION,
    key_added="leiden",
    flavor="igraph",
    directed=False,
)
print(adata_sc.obs["leiden"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.embedding(adata_sc, basis="X_umap", color="sample", ax=axes[0], show=False, title="colored by sample")
sc.pl.embedding(adata_sc, basis="X_umap", color="leiden", ax=axes[1], show=False, title=f"Leiden (resolution={CHOSEN_RESOLUTION})")
fig.tight_layout()
plt.show()

### Marker genes of each cluster

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 11  -  Differential expression between clusters
# ==========================================================================
# What to do:
#   1. One-vs-rest differential expression between the Leiden clusters of adata_sc, Wilcoxon test,
#      results stored under key "rank_genes_leiden"                          -> sc.tl.rank_genes_groups
#   2. Compute the dendrogram of the clusters on the PCA                     -> sc.tl.dendrogram
#   3. Dotplot of the top 5 genes of each cluster, scaled per gene            -> sc.pl.rank_genes_groups_dotplot
#      (arguments: groupby, key, n_genes, standard_scale="var")
# Expected: a dotplot with ~5 genes per cluster.

____
____
____

In [ ]:
# ==========================================================================
# RUN  |  The marker genes published in the paper
# Expected: the table of the paper's markers, then a dotplot (cell types of the paper x our clusters).
# Which clusters light up for which cell type?
# ==========================================================================
marker_table = pd.read_csv(MARKERS_CSV)
display(marker_table)

# dict {major cell type: [marker genes]} for the scRNA-seq data (genes missing from our data are dropped)
sc_major_markers = eh.markers_dict(marker_table, "scRNA-seq", "major", adata_sc)

sc.pl.dotplot(adata_sc, sc_major_markers, groupby="leiden", standard_scale="var", dendrogram=True)

In [ ]:
# ==========================================================================
# RUN  |  Save a checkpoint with the clusters
# Nothing to modify.
# ==========================================================================
adata_sc.write_h5ad(f"{MY}/adata_sc_leiden.h5ad")

### Cell type assignment - from clusters to cell types

Leiden clustering gives us a fine-grained partition of the data (about 15-20 clusters with the resolution we used) based purely on transcriptional similarity. These clusters are not yet cell types — they are just groups of cells that look alike to the algorithm.

Cell type assignment is a separate, manual step: for each cluster, we look at its marker genes (from the differential expression test) and known marker gene sets (e.g., from the literature) to decide which biological cell type it most likely represents.

It is expected that several clusters end up mapping to the **same** cell type (e.g., different activation states or conditions of the same cell population). This is a many-to-one relationship, not one-to-one,and the paper itself follows the same logic: 27 fine-grained clusters (full dataset, Louvain resolution 0.8) are grouped into only 9 major cell types based on marker genes. Our 17 clusters are at a comparable "fine" level, and we expect them to collapse into a similarly small number of major cell types once annotated.

In [ ]:
# ==========================================================================
# RUN  |  Top 5 marker genes per cluster, as a readable table
# Expected: 5 genes per cluster, ranked by log fold change.
# Keep this table at hand for the annotation.
# ==========================================================================
top5 = eh.top_markers_table(adata_sc, key="rank_genes_leiden", n=5)
print(top5.to_string())

### Your annotation

For **each** Leiden cluster you now have what you need to decide which cell type it is:
the top marker genes (table above), the dotplot of the paper's markers (above), and your knowledge of skin biology.
A reasonable workflow:

1. Read the top markers of the cluster. Do they match a known cell type? (Be careful: genes such as *RPS/RPL*, *B2M*, *TMSB4X*, or metallothioneins *MT1X/MT1G* are not specific; a cluster dominated by them may be low quality, stressed, or simply unspecific.)
2. Look at the paper's marker dotplot for the same cluster.
3. For ambiguous clusters, run a **targeted dotplot** with the markers of the candidate cell types (see the next task).
4. Write your label. It is perfectly normal that **several clusters map to the same cell type** (different states of the same population); and rare populations (for instance Schwann cells) may not form a cluster of their own at all.

The finer descriptions (basal, differentiated, wound-activated...) are *interpretations based on your knowledge* of skin biology: they are working hypotheses, not validated annotations.

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 12  -  Targeted check of ambiguous clusters
# ==========================================================================
# What to do (optional but recommended for clusters you are unsure about):
#   build a small dict {"group name": [marker genes]} with the genes of the candidate cell types
#   (e.g. T/NK cells, or any population you suspect), and plot it per cluster with sc.pl.dotplot.
# Expected: a dotplot in which one cluster stands out for the genes you chose.

targeted_markers = {
    # "T/NK markers": ["CD3D", "CD3E", "NKG7"],     # <- example of the format
}

____

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 13  -  Assign a cell type to every cluster
# ==========================================================================
# What to do: complete the dictionary {cluster number (as a string): cell type} for ALL clusters.
# Suggested labels (the major cell types of the paper that are expected in this subset):
#   "Keratinocytes", "Fibroblasts", "Myeloid cells", "Lymphoid cells", "Mast cells",
#   "Melanocytes", "Pericytes and smooth muscle cells", "Endothelial cells"
# You may use another label if you can justify it.
# Expected: a UMAP coloured by cell type, with a handful of well-separated groups.

cluster_to_celltype = {
    # "0": "...",
}

missing = sorted(set(adata_sc.obs["leiden"].unique()) - set(cluster_to_celltype))
assert not missing, f"Clusters without annotation: {missing}"

adata_sc.obs["cell_type"] = adata_sc.obs["leiden"].map(cluster_to_celltype).astype("category")
sc.pl.embedding(adata_sc, basis="X_umap", color="cell_type", title="Cell type annotation")

#### Validating the annotation against the published dataset

The cluster annotation above was built by manually inspecting marker gene dot plots, that is a reasonable first pass, but ultimately a subjective judgment call. Because our cells are a subset of the same GEO dataset analyzed in the original publication, we can perform an objective, independent check: the authors already assigned a cell type to every cell they kept (`GSE241132_cell_metadata.txt`, column `newMainCellTypes`), so we can match our own cells back to theirs by barcode and compare.

The two barcode naming conventions differ (`{donor}{timepoint}_{barcode}` in the paper's metadata vs. `{barcode}-{timepoint}` in ours), so we first reconcile the format, then merge in the published cell type label for every cell we can match. Finally, we cross-tabulate our Leiden clusters against these published labels: for each cluster, this tells us exactly how its cells were classified by the original authors, giving us a quantitative measure of how well our unsupervised clustering + marker-based annotation agrees with ground truth.

In [ ]:
# ==========================================================================
# READ  |  Read the metadata published by the authors
# Nothing to modify. Compare the two barcode formats: they differ, hence the next cell.
# ==========================================================================
paper_meta = pd.read_csv(PAPER_META, sep="\t", index_col=0)

print("Paper metadata barcode format (first 5):")
print(paper_meta.index[:5].tolist())
print("\nYour adata_sc barcode format (first 5):")
print(adata_sc.obs_names[:5].tolist())
print("\nPaper metadata columns:", paper_meta.columns.tolist())

In [ ]:
# ==========================================================================
# READ  |  Match our cells to the published annotation and cross-tabulate
# Nothing to modify. Read the function to_paper_barcode.
# The cross-tabulation: rows = YOUR clusters, columns = the cell types assigned by the authors.
# ==========================================================================
DONOR = "PWH27"

def to_paper_barcode(adata_barcode, donor=DONOR):
    core, tp = adata_barcode.rsplit("-", 1)  # "AAACCCAAGTATCCTG-1", "D0"
    return f"{donor}{tp}_{core}"

adata_sc.obs["paper_barcode"] = [to_paper_barcode(bc) for bc in adata_sc.obs_names]

paper_meta_pwh27 = paper_meta[paper_meta.index.str.startswith(DONOR)]
adata_sc.obs["paper_newMainCellTypes"] = adata_sc.obs["paper_barcode"].map(paper_meta_pwh27["newMainCellTypes"])

n_matched = adata_sc.obs["paper_newMainCellTypes"].notna().sum()
print(f"Matched {n_matched} / {adata_sc.n_obs} cells to the paper's published annotation")

# Cross-tabulation: your leiden clusters vs. the paper's own cell type calls
pd.crosstab(adata_sc.obs["leiden"], adata_sc.obs["paper_newMainCellTypes"])

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 14  -  Do you confirm or correct your annotation?
# ==========================================================================
# What to do:
#   Look at the cross-table above, cluster by cluster. For each cluster, which cell type did the authors
#   assign to most of its cells? Does it agree with your label? (Names differ slightly: "Keratinocyte"
#   vs "Keratinocytes", etc.)
#   Optionally, quantify the agreement with ONE number of your choice (e.g. the fraction of cells in
#   each cluster that falls in its majority published type), using the cross-table.
#   If you find a clear disagreement, go back to your dictionary above, correct it, and re-run.

# write your code here

**Question 9.** Which clusters, if any, disagree with the published annotation? Was it a mistake in your annotation, or is the published label a different level of granularity?

*Your answer (double-click to edit):* 

In [ ]:
# ==========================================================================
# RUN  |  Save a checkpoint with the annotation
# Nothing to modify.
# ==========================================================================
adata_sc.write_h5ad(f"{MY}/adata_sc_leiden_annotated.h5ad")

# --- SAFETY NET: instructor's annotated object (uncomment if you are stuck; cluster numbers are the instructor's!) ---
# adata_sc = sc.read_h5ad(f"{PROC}/adata_sc_leiden_res0.2_annotated.h5ad")

### A finer label for the deconvolution: cell states

Cell2location needs a reference with **one label per cell state**, finer than the major cell type (for instance, *basal*, *differentiated* or *wound-activated* keratinocytes). The next step labels each cluster with a **state**.

**Important:** the deconvolution that we will read later was computed by the instructors with a fixed list of 17 cell states. Your labels must come from this list, so that your results can be compared with it.

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 15  -  Assign a cell state to every cluster
# ==========================================================================
# What to do: complete {cluster number (string): cell state}, using ONLY the labels of this list.
# Think about which marker genes tell the states apart (you may need extra targeted dotplots).
CELL_STATES = [
    "Keratinocytes_basal_A", "Keratinocytes_basal_B", "Keratinocytes_differentiated",
    "Keratinocytes_suprabasal", "Keratinocytes_ribosomal_stress", "Keratinocytes_wound_activated",
    "Myeloid_MHCII_high", "Macrophages", "Lymphoid_T_NK", "Mast_cells",
    "Fibroblasts_CFD", "Fibroblasts_myofibroblast_like", "Fibroblasts_ECM_high",
    "Melanocytes", "Pericytes_SMC", "Endothelial_lymphatic", "Endothelial_vascular",
]
# Several clusters may share a state; some states may be absent from your clustering.
# Hint: the state must be coherent with the cell type you gave to the same cluster.

cluster_to_state = {
    # "0": "...",
}

missing = sorted(set(adata_sc.obs["leiden"].unique()) - set(cluster_to_state))
unknown = sorted(set(cluster_to_state.values()) - set(CELL_STATES))
assert not missing, f"Clusters without a state: {missing}"
assert not unknown, f"Labels not in CELL_STATES: {unknown}"

adata_sc.obs["cell_state"] = adata_sc.obs["leiden"].map(cluster_to_state).astype("category")
print(adata_sc.obs["cell_state"].value_counts())

### Coming back to the Visium 10x data

#### Annotating the spatial clusters with the paper's marker genes

We now turn to the spatial data. As we did for the single-cell data, we look at which genes distinguish each spatial cluster (differential expression, one-vs-rest) and compare them with the marker genes the authors used to annotate their spatial clusters (stored in `paper_marker_genes.csv`, ST-seq section). The paper follows the same logic: spot clusters were annotated based on their differentially expressed genes and on markers of distinct cell types from the scRNA-seq data.

One important difference with the single-cell case: a Visium spot is not a single cell but a mixture of several cells (about 55 µm in diameter). A spatial cluster should therefore be read as a group of spots *dominated* by a given cell type or tissue structure, not as a pure population. This is why the resulting label is called `spot_type`.

This annotation is also useful before moving to deconvolution. The deconvolution can only distribute spots among the cell types present in the single-cell reference. Structures that are not represented there (for example sweat glands, sebaceous glands or hair follicles, which appear among the spatial clusters of the paper) would be forced onto the closest available cell type. Looking at the spatial markers directly tells us where the reference is incomplete.

In [ ]:
# ==========================================================================
# RUN  |  Load your clustered spatial data
# Nothing to modify.
# ==========================================================================
sdata_all = sd.read_zarr(f"{MY}/spatial_data_clustered.zarr")
# --- SAFETY NET (uncomment if you are stuck; cluster numbers are the instructor's!) ---
# sdata_all = sd.read_zarr(f"{PROC}/spatial_data_clustered.zarr")
adata = sdata_all["table"]

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 16  -  Differential expression between spatial clusters
# ==========================================================================
# What to do: the same analysis you did for the single-cell clusters, now on the spatial clusters.
#   1. Print how many spots are in each cluster.
#   2. One-vs-rest Wilcoxon DE between the Leiden clusters of `adata`, stored under key "rank_genes_leiden".
#   3. Print the top 5 genes per cluster as a readable table (helper: eh.top_markers_table).
#   4. Dendrogram + dotplot of the top 5 genes per cluster (same functions as for the single cells).
# Expected: a table with 5 genes per cluster and a dotplot.

adata = sdata_all["table"]
____
____
____
____

In [ ]:
# ==========================================================================
# RUN  |  The marker genes of the paper on the spatial clusters
# Expected: a dotplot (the paper's spatial marker categories x your spatial clusters).
# ==========================================================================
marker_table = pd.read_csv(MARKERS_CSV)
st_markers = eh.markers_dict(marker_table, "ST-seq", "major", adata)

sc.pl.dotplot(adata, st_markers, groupby="leiden", standard_scale="var", dendrogram=True)

#### Spot-type labels: an interpretation

You will now give each spatial cluster a name, from its top DE markers, the paper's ST marker categories, the tissue image and the position of the clusters on the sections (use the maps above). Remember that a spot is a **mixture** of cells: a cluster is *dominated by* a cell type or a tissue structure, it is not a pure population.

Your `spot_type` labels are **working interpretations, not validated annotations**. In particular, think about which clusters you are unsure about: they are good questions for the deconvolution!

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 17  -  Name the spatial clusters
# ==========================================================================
# What to do: complete {cluster number (string): short descriptive name} for ALL spatial clusters.
# Examples of the kind of names you can use (they are NOT the answer, they only show the format):
#   "Suprabasal_epidermis", "Dermis_fibroblasts", "Inflamed_tissue"
# Think about: epidermis vs dermis, wound edge vs intact tissue, immune infiltrate, vessels...

cluster_to_spot_type = {
    # "0": "...",
}

missing = sorted(set(adata.obs["leiden"].unique()) - set(cluster_to_spot_type))
assert not missing, f"Clusters without a name: {missing}"

adata.obs["spot_type"] = adata.obs["leiden"].map(cluster_to_spot_type).astype("category")
adata.obs[["leiden", "spot_type"]].drop_duplicates().sort_values("leiden")

**Question 10.** For which clusters are you least sure of your label, and why? What additional information would help you decide?

*Your answer (double-click to edit):* 

### Spatial deconvolution with Cell2location

Annotating spatial clusters tells us what each region looks like, but not which cell types, and in what proportion, are present in each spot. For this we use **Cell2location**, a Bayesian model that decomposes the gene counts of each spot into contributions from the cell types of a single-cell reference. This is the step that transfers the single-cell annotation to the spatial data. It is the same tool used in the original publication.

It works in two stages:

1. **Reference signatures.** Using the raw counts of the single-cell data and our annotation (`cell_state`, one label per Leiden cluster), a regression model estimates the average expression profile of each cell state.
2. **Spatial mapping.** Using the raw counts of the spatial data, the model estimates how many cells of each state are present in each spot. We follow the paper's parameters: `N_cells_per_location=20` (the expected number of cells per spot, estimated from nuclei counts on the H&E images) and `detection_alpha=20` (which accounts for large differences in RNA detection sensitivity between spots).

Both stages require **raw counts**, not normalized values, which is why we saved a `counts` layer before normalization. We use the 5% quantile of the posterior distribution (`q05`) as the abundance estimate, a conservative value that represents high confidence. We copy these estimates into `.obs`, one column per cell state, so that they can be plotted on the tissue like any other variable. We also derive a discrete label (`dominant_state`), the state with the highest abundance in each spot. This is only a convenient summary of the deconvolution, not an independent annotation.

Two practical remarks. Training is computationally heavy (a GPU is strongly recommended for such an analysis). And since the model can only assign cells to states present in the reference, spots made of structures missing from the single-cell data will be assigned to the closest available state and should be interpreted with care.

**In this practical you do not train the model**: training takes hours on a CPU (a GPU is strongly recommended) and it is better run from a script than from a notebook. The instructors ran it for you and provide the output table, `q05_cell_abundance.csv`. Below, the code that was run is **shown in full** (not executable on purpose): read it and make sure you understand each step. The curious can find the standalone script `run_cell2location.py` in the course repository.

#### The code that was run (read only)

**Setup**
```python
import numpy as np
import pandas as pd
import torch
import scvi
import matplotlib.pyplot as plt
from cell2location.utils.filtering import filter_genes
from cell2location.models import RegressionModel, Cell2location

N_THREADS = 8                  # number of cores you can use
REF_EPOCHS = 250
VIS_EPOCHS = 5000              # use ~200 for a quick demo

torch.set_num_threads(N_THREADS)
scvi.settings.seed = 0
ACC = "gpu" if torch.cuda.is_available() else "cpu"
```

**Stage 1 - reference signatures from the scRNA-seq (raw counts, labels = `cell_state`)**
```python
adata_ref = adata_sc.copy()
adata_ref.X = adata_ref.layers["counts"].copy()

# Keep informative genes only
selected = filter_genes(adata_ref, cell_count_cutoff=5,
                        cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)
adata_ref = adata_ref[:, selected].copy()

RegressionModel.setup_anndata(adata_ref, batch_key="sample", labels_key="cell_state")
mod_ref = RegressionModel(adata_ref)
mod_ref.train(max_epochs=REF_EPOCHS, accelerator=ACC)
adata_ref = mod_ref.export_posterior(
    adata_ref, sample_kwargs={"num_samples": 1000, "batch_size": 2500, "accelerator": ACC})

# Average expression signature of each cell state
states = adata_ref.uns["mod"]["factor_names"]
inf_aver = adata_ref.varm["means_per_cluster_mu_fg"][
    [f"means_per_cluster_mu_fg_{s}" for s in states]].copy()
inf_aver.columns = states
```

**Stage 2 - mapping onto the Visium spots (raw counts of the spatial data)**
```python
adata_vis = adata.copy()
adata_vis.X = adata_vis.layers["counts"].copy()

# Use the genes shared by spatial data and signatures
shared = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, shared].copy()
inf_aver = inf_aver.loc[shared, :].copy()

Cell2location.setup_anndata(adata_vis, batch_key="sample")
mod_vis = Cell2location(adata_vis, cell_state_df=inf_aver,
                        N_cells_per_location=20, detection_alpha=20)
mod_vis.train(max_epochs=VIS_EPOCHS, batch_size=None, train_size=1, accelerator=ACC)

adata_vis = mod_vis.export_posterior(
    adata_vis, sample_kwargs={"num_samples": 1000,
                              "batch_size": mod_vis.adata.n_obs, "accelerator": ACC})
q05 = adata_vis.obsm["q05_cell_abundance_w_sf"].copy()      # the table you will receive
```

**Question 11.** Why does Cell2location need *raw counts* (not normalised) for both the reference and the spatial data? What do `N_cells_per_location` and `detection_alpha` represent, and where does the value 20 come from?

*Your answer (double-click to edit):* 

**Question 12.** Stage 1 learns a *signature* per cell state, stage 2 only *uses* those signatures. What does this imply for a structure that is present in the tissue but absent from the single-cell reference (for example hair follicles or neutrophils)?

*Your answer (double-click to edit):* 

#### Why do we use the `q05` cell abundance?

Cell2location is a **Bayesian** model. For every spot and every cell state it does not return a single number, but a whole **posterior distribution** of plausible abundances (we draw 1000 samples from it with `export_posterior`). From this distribution the package stores several summaries, each with its own prefix:

| Prefix | Meaning |
|---|---|
| `means_` | mean of the posterior distribution |
| `stds_` | standard deviation |
| `q05_` | **5% quantile**: the abundance is above this value with ~95% probability |
| `q95_` | 95% quantile |

We work with `q05`, a **conservative** estimate. If the model is uncertain about a cell state in a spot, the distribution is wide and `q05` is pushed towards zero; if the model is confident, `q05` stays close to the mean. A high `q05` therefore means *"abundant and well supported by the data"*.

**This choice follows the original paper.** In the Methods of Liu et al. (*Spatiotemporal single-cell roadmap of human skin wound healing*, Cell Stem Cell 2025), section *"Deconvolution of ST-seq data and wound bulk RNA-seq data"*, the authors write:

> "The posterior distribution of cell abundance for each cell type in each spot was summarized as 5% quantile, representing high confidence, which was used for visualization and colocalization analysis."

The same section reports the two parameters that were changed from the defaults, which we also use:

- `N_cells_per_location = 20`: expected number of cells per spot, estimated by the authors from the average number of nuclei per spot in the H&E images.
- `detection_alpha = 20`: regularisation of the per-location normalisation, to account for large differences in RNA detection sensitivity between Visium spots.

#### Where the information comes from

- **From the paper (verified in its Methods):** the use of Cell2location, the 5% quantile as "high confidence" summary, `N_cells_per_location=20`, `detection_alpha=20`.
- **General knowledge about Cell2location (not stated in this paper):** the meaning of the posterior summaries (`means_`, `stds_`, `q05_`, `q95_`) and the interpretation of `q05` as a conservative estimate. See Kleshchevnikov et al., *Nature Biotechnology* 2022.

#### Differences between this notebook and the paper

This notebook follows the paper's Cell2location settings, but it is **not an exact reproduction**:

- The paper trained the reference regression model on **each cell type's top 100 marker genes**; here we select genes with Cell2location's `filter_genes` function instead.
- Our reference has **17 cell states** derived from our own clustering (`cell_state`), not the paper's annotation.
- Depending on the run, we may have used fewer training epochs and a subsampled reference for speed.

Expect similar *patterns*, not identical numbers.

In [ ]:
# ==========================================================================
# RUN  |  Save the annotated single-cell object
# Nothing to modify.
# ==========================================================================
adata_sc.write_h5ad(f"{MY}/adata_sc_annotated_cell_state.h5ad")

#### From cell types to tissue: reading the Cell2location results

Cell2location combined the two datasets: the **cell-state signatures learned from scRNA-seq** were used to estimate, **in every Visium spot, how many cells of each state** are present. The result is a table with one row per spot and one column per cell state.

It does not matter how the model was run (external script or directly in the notebook): we start from the same table, `q05_cell_abundance.csv`. We add its columns to `adata.obs` with the prefix `c2l_`, and we also store the most abundant state per spot (`dominant_state`) as a simple discrete label.

We use the **`q05` summary**, i.e. the 5% quantile of the posterior distribution of each abundance. This is a conservative estimate that the original paper uses as "high confidence" (see the previous section). Finally we re-declare the table to the `SpatialData` object, so that we can plot these columns on the tissue images.

In [ ]:
# ==========================================================================
# RUN  |  Load the Cell2location results into the spatial table
# Nothing to modify. Expected: a table with one row per spot and one column per cell state.
# ==========================================================================
adata = sdata_all["table"]

q05 = pd.read_csv(Q05_FILE, index_col=0)          # spots x cell states, provided by the instructors
assert (q05.index == adata.obs_names).all(), "Spot names differ: use the spatial table of the instructors' checkpoint."

for state in q05.columns:
    adata.obs[f"c2l_{state}"] = q05[state].values          # one obs column per cell state
adata.obs["dominant_state"] = q05.idxmax(axis=1).astype("category")   # the most abundant state in each spot

adata = eh.relink_table(sdata_all, adata, SAMPLES)         # re-declare the table -> spots link
print(q05.shape)
q05.head()

In [ ]:
# ==========================================================================
# RUN  |  Save the spatial object with the Cell2location results
# Nothing to modify (checkpoint, can take a minute).
# ==========================================================================
sdata_all.write(f"{MY}/spatial_data_annotated_with_cell2location.zarr", overwrite=True)

We will draw many maps, so we define a small helper that shows one variable (an `obs` column or a gene) on the four tissue sections. Optionally, all sections can share the same color scale (`vmax`): this is essential when we compare timepoints, otherwise every panel would be stretched to its own range and a tiny abundance in healthy skin would look as strong as a large one in the wound.

In [ ]:
# ==========================================================================
# RUN  |  Helper: one variable on the four tissue sections
# Nothing to modify: from now on, spatial_row(variable, axes) draws a variable on the 4 sections.
# ==========================================================================
def spatial_row(color, axes, vmax=None, cmap="viridis"):
    """Draw one variable (an .obs column or a gene) on the 4 tissue sections.
    vmax: if given, all panels share the colour scale [0, vmax]."""
    eh.spatial_row(sdata_all, color, axes, SAMPLES, vmax=vmax, cmap=cmap)

#### Step 1: does the deconvolution make sense?

Before interpreting any result, we check the model with three simple controls.

##### 1a. How many cells per spot, and where?

Summing `q05` over all cell states gives the **estimated number of cells in each spot**. The model started from a prior expectation of about 20 cells per spot (`N_cells_per_location=20`, estimated by the authors of the paper from the nuclei in the H&E images). Because `q05` is conservative, the sum may be lower than that. What matters is that cell density changes **in space** and **across samples**, and a map shows it immediately.

We also compare this estimate with the sequencing depth. A strong correlation tells us that the model is sensitive to depth. This is a limitation to keep in mind, not necessarily an error: denser tissue really does contain more cells and more RNA.

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 18  -  1a. How many cells per spot, and where?
# ==========================================================================
# What to do:
#   1. Store in adata.obs["c2l_total_cells"] the sum of q05 over the cell states of each spot.
#   2. Map it on the 4 sections with spatial_row (figure with 1 row x 4 columns).
#   3. Compute the Spearman correlation with the sequencing depth (adata.obs["total_counts"]) -> spearmanr.
# Expected: a map of cell density, and a printed correlation.

____
fig, axes = plt.subplots(1, len(SAMPLES), figsize=(5 * len(SAMPLES), 5))
____
fig.suptitle("Estimated number of cells per spot (sum of q05)")
fig.tight_layout()
plt.show()

____
print(f"Spearman(total estimated cells, total_counts) = {rho:.2f}")

**Question 13.** Does the estimated cell density change in space and across samples? Is the correlation with sequencing depth high? Is that necessarily a problem?

*Your answer (double-click to edit):* 

#### 1b. Where is each cell state? (same color scale in all samples)

This is where spatial transcriptomics shows its strength: we can see **where** a cell type is located in the tissue, and how this changes from healthy skin to the wound at different timepoints. In scRNA-seq the same information would be reduced to a proportion.

Each row below is one cell state, each column one sample. The color scale is **shared across the four samples** (the upper limit is the 99th percentile of that state), so that panels can be compared.

Look at the maps with these questions in mind: is the state located where we expect from the tissue image (epidermis on top, dermis below, wound edge)? Does it appear or disappear over time? A state that looks uniformly spread everywhere is often a sign that the model cannot distinguish it well.

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 19  -  1b. Where is each cell state? (shared colour scale)
# ==========================================================================
# What to do:
#   Choose 4-5 cell states that you find interesting (the list of available states is printed below),
#   and map each of them on the 4 sections, one row per state, using a colour scale shared by the
#   4 samples: vmax = 99th percentile of that state (q05[state].quantile(0.99)).
# Expected: a grid (states x samples). The plotting loop is written for you.

print(list(q05.columns))

states_to_show = [
    # "...",
]
assert len(states_to_show) > 0, "Choose at least one state"

fig, axes = plt.subplots(len(states_to_show), len(SAMPLES), figsize=(5 * len(SAMPLES), 5 * len(states_to_show)), squeeze=False)
for row, state in enumerate(states_to_show):
    vmax = q05[state].quantile(0.99)           # shared scale across the 4 samples
    spatial_row(f"c2l_{state}", axes[row], vmax=vmax)
    axes[row, 0].set_ylabel(state, fontsize=12)
fig.tight_layout()
plt.show()

**Question 14.** Pick two states: where are they located, does it match the tissue (epidermis on top, dermis below, wound edge)? Do they appear or disappear over time? Did you see a state that looks uniformly spread everywhere, and what could that mean?

*Your answer (double-click to edit):* 

### 1c. An independent check with genes

This is the most convincing control. The signatures of the cell states come from **scRNA-seq**, but the gene expression we measured in each spot is an **independent measurement** made on the spatial data. If the deconvolution works, the abundance of a cell state must correlate with the expression of one of its marker genes **in the same spots**.

You will test a few (cell state, marker gene) pairs of your choice, and then look at the same comparison on the tissue: abundance on top, gene expression below. If the two maps overlap, the deconvolution is capturing real biology.

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 20  -  1c. An independent check with genes
# ==========================================================================
# What to do:
#   Propose (cell state, marker gene) pairs, with genes NOT used to define the states (use the paper's table of
#   markers, or your own knowledge). For each pair, compute the Spearman correlation between the abundance
#   of the state (q05[state]) and the expression of the gene (adata.obs_vector(gene)) in the same spots.
#   One example is given. Add at least 3 more.
#   Then look at the same comparison on the tissue: abundance (top row) vs gene (bottom row).
# Expected: a table of correlations, and a figure with abundance and gene maps that overlap if the model works.

checks = [
    ("Keratinocytes_wound_activated", "KRT6A"),     # example given
    # ("...", "..."),
]

rows = []
for state, gene in checks:
    if gene not in adata.var_names:
        print(f"{gene} not in the filtered gene set, skipped")
        continue
    rho, p = spearmanr(q05[state].values, adata.obs_vector(gene))
    rows.append({"state": state, "marker_gene": gene, "spearman_rho": round(rho, 2)})
print(pd.DataFrame(rows).to_string(index=False))

# The same thing on the tissue, for the first pair: abundance (top) vs gene expression (bottom)
state, gene = checks[0]
fig, axes = plt.subplots(2, len(SAMPLES), figsize=(5 * len(SAMPLES), 10))
spatial_row(f"c2l_{state}", axes[0], vmax=q05[state].quantile(0.99))
spatial_row(gene, axes[1])
axes[0, 0].set_ylabel(f"Cell2location q05\n({state})", fontsize=11)
axes[1, 0].set_ylabel(f"{gene} expression", fontsize=11)
fig.tight_layout()
plt.show()

**Question 15.** Which pairs correlate well and which do not? For the ones that do not, can you think of a biological or technical reason?

*Your answer (double-click to edit):* 

#### Step 2: linking the two annotations (ST clusters × cell states)

So far we have two descriptions of the spatial data that were built **independently**: the ST clusters, named from their marker genes (`spot_type`), and the cell states, which come from the single-cell reference. For every ST cluster we now compute the mean abundance of each cell state, and we read the result in two complementary ways:

- **Left, composition**: "what is each cluster made of?" Each row is normalized to sum to 1.
- **Right, concentration**: "where is each cell state concentrated?" Each column is rescaled between 0 and 1, which makes rare states visible too.

This figure is also the moment to go back to the **tentative labels** you gave to the spatial clusters, especially the ones you were unsure about. They now have an estimated cellular composition. Do you think it confirms the name you gave them, or does it suggest another one? Keep in mind that the reference contains no neutrophils, so a cluster rich in neutrophils will look as if it were made of the closest available state.

In [ ]:
# ==========================================================================
# RUN  |  Mean abundance of each cell state in each spatial cluster
# Expected: two heatmaps (composition on the left, concentration on the right), one row per spatial cluster.
# ==========================================================================
composition, concentration = eh.plot_cluster_by_state(q05, adata, cluster_key="leiden", name_key="spot_type")

**Question 16.** Does the cellular composition of each cluster confirm the name you gave it, or suggest another one? Focus on the clusters you were unsure about. (Remember: the reference contains no neutrophils.)

*Your answer (double-click to edit):* 

#### Step 3: how does the composition change over time?

Now we add the **time** dimension. For every sample (Skin, Wound1, Wound7, Wound30) we average the abundance of each cell state and show it as a proportion. This is the spatial version of the cell-proportion plots usually made with scRNA-seq, so we can put the two side by side.

The two bars **are not expected to match**, and the reasons are a good discussion topic:

- In scRNA-seq, tissue dissociation favours some cell types and loses others.
- In spatial data every spot captures the tissue as it is, but `q05` is a conservative estimate, and the depth of the sample Skin is much lower than in the other samples.

Neither of them is "the truth": they are two different views of the same tissue.

In [ ]:
# ==========================================================================
# RUN  |  Cell-state proportions per sample: spatial vs single-cell
# Expected: the mean number of cells per spot per sample, then two stacked bar plots (ST left, scRNA-seq right).
# ==========================================================================
st_prop, sc_prop = eh.compare_state_proportions(q05, adata, adata_sc, SAMPLES)

**Question 17.** Describe how the composition changes from Skin to Wound30 in the two datasets. Where do they agree, where do they differ, and why?

*Your answer (double-click to edit):* 

#### Step 4: who is next to whom?

Until now we looked at each spot on its own. Now we use what only spatial data can give us: **position**. We work on **one sample at a time**, because spots from different sections are not neighbours. We start with `Wound7` (one of the richest samples); change `S` to look at the others.

First we build a **spatial graph**: on a Visium slide the spots are arranged on a hexagonal grid, so every spot has 6 neighbours (`coord_type="grid"`, `n_neighs=6`).

In [ ]:
# ==========================================================================
# READ  |  Spatial graph for one sample
# Nothing to modify. Change S to look at other samples later on.
# ==========================================================================
S = "Wound7"
a = adata[adata.obs["sample"] == S].copy()
a.obs["spot_type"] = a.obs["spot_type"].cat.remove_unused_categories()

# Build the spatial graph: each Visium spot has 6 neighbours on the hexagonal grid
sq.gr.spatial_neighbors(a, coord_type="grid", n_neighs=6, spatial_key="spatial")

#### 4a. Neighborhood enrichment between ST clusters

We count how often two types of spots are adjacent, and compare this number with what we would expect if the labels were **randomly shuffled** over the tissue (permutation test). A high positive value means "these two types are neighbours more often than by chance"; a strongly negative value means "they avoid each other".

In [ ]:
# ==========================================================================
# YOUR TURN  |  Task 21  -  4a. Neighborhood enrichment between spatial clusters
# ==========================================================================
# What to do:
#   Run the permutation test (sq.gr.nhood_enrichment, cluster_key="spot_type", seed=0) on the object `a`,
#   then plot the result (sq.pl.nhood_enrichment, same cluster_key, figsize=(7, 6)).
# Expected: a heatmap of z-scores: positive = neighbours more often than by chance, negative = avoidance.

____
____

**Question 18.** Which pairs of spot types are strongly enriched as neighbours, and which avoid each other? Does it match the tissue architecture (epidermis / dermis)?

*Your answer (double-click to edit):* 

#### 4b. Spatial co-abundance of cell states

Here we ignore the clusters and look directly at the **abundances**. Two cell states that tend to be abundant in the same spots are probably close to each other in the tissue. This is the intuition that the paper formalises with non-negative matrix factorization (NMF) to find "microenvironments". The hierarchical clustering of the heatmap puts next to each other the states with similar spatial profiles.

In [ ]:
# ==========================================================================
# READ  |  Spatial co-abundance of cell states
# Nothing to modify. Look at how the hierarchical clustering groups the cell states.
# ==========================================================================
corr = q05.loc[(adata.obs["sample"] == S).values].corr(method="spearman")
sns.clustermap(corr, cmap="vlag", center=0, vmin=-1, vmax=1, figsize=(10, 10))
plt.suptitle(f"Spatial co-abundance of cell states ({S})", y=1.02)
plt.show()

### 4c. How spatially structured is each cell state?

**Moran's I** measures whether similar values are close to each other (values near 1) or scattered at random (values near 0). Applied to the abundances, it separates the states that form precise structures, such as an epidermal layer, from those that are diffuse everywhere. This is information that scRNA-seq cannot provide, because it does not know where the cells were.

In [ ]:
# ==========================================================================
# READ  |  Moran's I of each cell state
# Nothing to modify. Which states form precise structures and which are diffuse?
# ==========================================================================
state_cols = [f"c2l_{c}" for c in q05.columns]
sq.gr.spatial_autocorr(a, attr="obs", genes=state_cols, mode="moran", n_perms=100, seed=0)

moran = a.uns["moranI"]["I"].sort_values()
moran.index = [i.replace("c2l_", "") for i in moran.index]
moran.plot(kind="barh", figsize=(7, 6))
plt.xlabel("Moran's I (spatial autocorrelation of the abundance)")
plt.title(f"Which cell states form spatial structures? ({S})")
plt.tight_layout()
plt.show()

**Question 19.** Which cell states are the most and the least spatially structured? Is it what you expected from their biology?

*Your answer (double-click to edit):* 

## Take-home message and caveats

Each analysis answers a question that the previous one could not:

- **scRNA-seq** told us *who* is there (cell states).
- **Deconvolution** told us *where* they are (Step 1).
- **The cluster × state heatmap** linked *where* to *what we called that region* (Step 2).
- **The timepoints** told us *when* (Step 3).
- **The neighbourhood analysis** told us *next to whom* (Step 4): this is the only question that really needs spatial coordinates.

None of these questions can be answered with dissociated cells alone. This is what spatial transcriptomics adds.

**Limitations to keep in mind**

- `q05` is a conservative estimate: the sum per spot can be lower than the true number of cells, and proportions differ from those measured in scRNA-seq.
- The reference contains no neutrophils, glands or hair follicle: spots made of these structures are assigned to the closest available state.
- The sample `Skin` has a much lower sequencing depth than the others, which affects both the estimated number of cells and the comparisons across timepoints.
- If the model was run in a reduced version (fewer epochs, subsampled reference), the values can differ from the full run.
- Your `spot_type` labels are working interpretations: the deconvolution helps to test them, but it does not prove them.